# Machine Learning From Scratch
**Author:** Mohamed Ahmed Beder <br>
**GitHub:** [Beder-asu](https://github.com/Beder-asu) <br>
**LinkedIn:** [mobeder88](https://linkedin.com/in/mobeder88) <br>
**Email:** mobeder88@gmail.com


# Dataset
The Iris Dataset Overview
The dataset contains 150 samples of iris flowers, divided equally among 3 different species (50 samples per species). The goal is typically to train a model to predict the species of an iris flower based on its physical measurements.

1. Features (The X variable)
For each flower, four physical characteristics (features) were measured in centimeters. In your code, X is a 2D NumPy array with 150 rows and 4 columns corresponding to these measurements:

- Sepal length (sepal_length)
- Sepal width (sepal_width)
- Petal length (petal_length)
- Petal width (petal_width)

2. Target/Labels (The y variable)
The dataset includes the actual species of each flower. In its raw form, these are strings. Your code maps these string names into integer labels to make them usable for standard mathematical machine learning models:

- setosa → 0
- versicolor → 1
- virginica → 2

The resulting y variable is a 1D NumPy array of length 150 containing these integer labels.

Summary
In short, this dataset gives you 150 data points, where the input (X) consists of 4 numerical measurements of a flower, and the output (y) is a single number representing which of the 3 iris species that flower belongs to. It's an excellent introductory dataset for classification algorithms like Logistic Regression, K-Nearest Neighbors, or Support Vector Machines.



In [1]:
import numpy as np
import plotly.express as px

df = px.data.iris()
X = df[['sepal_length','sepal_width','petal_length','petal_width']].values
y = df['species'].map({'setosa':0,'versicolor':1,'virginica':2}).values
df 

,sepal_length,sepal_width,petal_length,petal_width,species,species_id
0,5.1,3.5,1.4,0.2,setosa,1
1,4.9,3.0,1.4,0.2,setosa,1
2,4.7,3.2,1.3,0.2,setosa,1
3,4.6,3.1,1.5,0.2,setosa,1
4,5.0,3.6,1.4,0.2,setosa,1
...,...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica,3
146,6.3,2.5,5.0,1.9,virginica,3
147,6.5,3.0,5.2,2.0,virginica,3
148,6.2,3.4,5.4,2.3,virginica,3


In [2]:
df.to_csv('iris_dataset.csv', index=False)

In [3]:


# --- Pairwise Scatter Matrix ---
fig = px.scatter_matrix(
    df,
    dimensions=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'],
    color='species',
    title='Iris Dataset — Pairwise Feature Scatter (Linear Separability Check)',
    opacity=0.7
)
fig.update_layout(width=900, height=900)
fig.show()

# --- 3D Scatter (best 3 features) ---
fig_3d = px.scatter_3d(
    df,
    x='petal_length',
    y='petal_width',
    z='sepal_length',
    color='species',
    title='Iris 3D View — Petal Length vs Petal Width vs Sepal Length',
    opacity=0.8
)
fig_3d.show()


---

# Statistical Models

A **statistical model** is a mathematical representation that describes the relationship between variables in a dataset. It aims to approximate the true mechanism that generated the data by capturing patterns while explicitly accounting for uncertainty and randomness. 

At its core, a statistical model is based on probability distributions and mathematically formalizes the connection between inputs (independent variables/features) and outputs (dependent variables/targets).

### Key Components of a Statistical Model

1. **Variables**: 
   - **Predictors (Features/Inputs/ $X$)**: The data you have (e.g., house size, location).
   - **Responses (Targets/Outputs/ $y$)**: The outcome you're trying to predict or understand (e.g., house price).

2. **Parameters ( $\theta$ or $\beta$ )**: 
   These are the internal "weights" or constants in the model that determine how much influence each variable has. The goal of "training" or "fitting" the model is to estimate the best values for these parameters from your data.

3. **Deterministic Structure**:
   The mathematical equation mapping inputs to outputs. For example, a simple linear model: $y = \beta_0 + \beta_1 X$

4. **Stochastic Component (Error Term/ $\epsilon$ )**:
   Unlike pure mathematical equations (like $Force = mass \times acceleration$), real-world data is messy. A statistical model includes an error term to account for randomness, unmeasured variables, or "noise." 
   The full framework looks like this: $y = \overbrace{f(X, \beta)}^{\text{Signal}} + \overbrace{\epsilon}^{\text{Noise}}$

### Why Do We Use Statistical Models?
According to statistician George Box, *"All models are wrong, but some are useful."* We use them primarily for two reasons:

1. **Inference (Understanding)**: To understand the relationships between variables. (e.g., "Does increasing advertising spend by $1M lead to higher sales, and if so, by how much exactly?")
2. **Prediction (Forecasting)**: To predict future or unobserved outcomes based on new data. (e.g., "Given the weather today, what is the probability it will rain tomorrow?")

### Common Examples
- **Linear Regression**: Models a straight-line relationship between variables.
- **Logistic Regression**: Models the probability of a binary outcome (e.g., Spam or Not Spam).
- **Support Vector Machine (SVM)**: Maximizes the margin between classes points using constraints.

____

## Linear Regression

**Description:** Linear Regression models the relationship between input features $X$ and a continuous output $y$ by fitting a straight line (or hyperplane in higher dimensions) that minimizes the total squared error.

**How it works:**
1. Prepend a column of ones to $X$ to handle the bias/intercept term ($\beta_0$)
2. Initialize weight vector $\mathbf{w}$ to zeros
3. Iteratively update $\mathbf{w}$ using **Gradient Descent** to minimize the **Mean Squared Error (MSE)** loss

**Loss Function — Mean Squared Error:**

$$\text{MSE} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}_i - y_i)^2 = \frac{1}{m}\|\tilde{X}\mathbf{w} - \mathbf{y}\|^2$$

**Gradient (derivative of MSE w.r.t. weights):**

$$\nabla_w = \frac{1}{m} \tilde{X}^T (\tilde{X}\mathbf{w} - \mathbf{y})$$

**Update Rule:**

$$\mathbf{w} \leftarrow \mathbf{w} - \alpha \cdot \nabla_w$$

where $\alpha$ is the learning rate, $m$ is the number of samples, and $\tilde{X}$ is $X$ with a prepended ones column.

**Prediction:** $\hat{y} = \tilde{X} \mathbf{w}$


In [2]:
class LinearRegression:

    def __init__(self, lr = 0.1, n_iter = 1000):
        self.lr = lr
        self.n_iter = n_iter

    def fit(self,X,Y):
        m,n = X.shape
        newX = np.hstack([np.ones((m,1)) , X])
        self.w = np.zeros(n+1)

        for _ in range(self.n_iter):
            dw = (1/m)*(newX.T)@(newX @ self.w - Y)  
            self.w -= self.lr * dw

    def predict(self, X):
        newX = np.hstack([np.ones((X.shape[0], 1)), X])
        return newX @ self.w

_____

## Logistic Regression

**Description:** Logistic Regression is a binary classification model that estimates the probability of an input belonging to class 1. It uses the same linear equation as Linear Regression, but wraps the output through a **Sigmoid function** to squish it into the range $[0, 1]$.

**How it works:**
1. Prepend a column of ones to $X$ for the bias term
2. Initialize weights $\mathbf{w}$ to zeros
3. Iteratively update $\mathbf{w}$ using Gradient Descent on the **Binary Cross-Entropy (Log Loss)**

**Sigmoid Function:**

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Predicted Probability:**

$$\hat{y} = \sigma(\tilde{X}\mathbf{w})$$

**Loss Function — Binary Cross-Entropy:**

$$\mathcal{L} = -\frac{1}{m}\sum_{i=1}^{m}\left[y_i \log(\hat{y}_i) + (1 - y_i)\log(1 - \hat{y}_i)\right]$$

**Gradient:**

$$\nabla_w = \frac{1}{m} \tilde{X}^T (\hat{y} - \mathbf{y})$$

**Final Prediction:** Class 1 if $\hat{y} \geq 0.5$, else Class 0.


In [3]:
class LogisticRegression:
    def __init__(self, lr, n_iter):
        self.lr = lr
        self.n_iter = n_iter

    def sigmoid(self,x):
        return 1/(1 + np.exp(-x) )   

    def fit(self, X,Y):
        m,n = X.shape
        newX = np.hstack([np.ones((m,1)) , X])
        self.w = np.zeros(n+1)

        for _ in range(self.n_iter):
            y = self.sigmoid (newX @ self.w)
            dw = (1/m)*(newX.T)@(y - Y)    
            self.w -=self.lr * dw  

    def prop(self, X):
        newX = np.hstack([np.ones((X.shape[0], 1)), X])
        return self.sigmoid(newX @ self.w)
    
    def predict(self,X):
        return (self.prop(X) >= 0.5)      

____

## Support Vector Machine (SVM)

**Description:** SVM finds the optimal hyperplane that separates two classes with the **maximum margin** — the widest possible gap between the closest data points of each class (called *support vectors*). Uses a soft-margin formulation to allow some misclassifications via penalty parameter $C$.

**How it works:**
1. Convert labels to $\{-1, +1\}$
2. Initialize weights $\mathbf{w}$ to zeros, bias $b$ to 0
3. For each iteration, identify margin-violating points and update $\mathbf{w}$ and $b$

**Decision Function:**

$$f(\mathbf{x}) = \mathbf{x} \cdot \mathbf{w} + b$$

**Hinge Loss (Soft-Margin) Objective:**

$$\min_{\mathbf{w}, b} \|\mathbf{w}\|^2 + C \sum_{i=1}^{m} \max(0,\ 1 - y_i(\mathbf{x}_i \cdot \mathbf{w} + b))$$

**Gradient Update (for margin-violating points where $y_i(\mathbf{x}_i \cdot \mathbf{w} + b) < 1$):**

$$\nabla_w = 2\mathbf{w} - C \sum_{i \in \text{violations}} y_i \mathbf{x}_i$$
$$\nabla_b = -C \sum_{i \in \text{violations}} y_i$$

**Prediction:** Class 1 if $f(\mathbf{x}) \geq 0$, else Class 0.


In [4]:
class SVM:
    def __init__(self, C=1.0, lr=0.001, n_iter=1000):
        self.C = C
        self.lr = lr
        self.n_iter = n_iter
        self.w = None
        self.b = 0

    def fit(self, X,Y):
        m, n = X.shape
        Y = np.where(Y == 1, 1, -1)
        self.w = np.zeros(n)
        for _ in range(self.n_iter):
            margin_pts = Y*(X@self.w + self.b) < 1    # soft margin
            dw = 2*self.w - self.C * np.dot(X[margin_pts].T , Y[margin_pts])
            db = -self.C * np.sum(Y[margin_pts])
            self.w -= self.lr * dw
            self.b -= self.lr * db

    def decision_function(self, X):
        return X @ self.w + self.b  

    def prop(self, X):
        return self.decision_function(X)    

    def predict(self, X):
        return (self.decision_function(X) >= 0).astype(int)          

---

# Current Approach

The **One-vs-Rest (OvR)** approach (also sometimes called **One-vs-All** or **OvA**) is a common strategy used to adapt binary classification algorithms (like Logistic Regression or Support Vector Machines) so they can handle **multi-class classification problems** (situations where you have more than two categories to predict).

Since many fundamental machine learning algorithms are designed to only answer "Yes" or "No", OvR provides a clever workaround to handle questions like "Is it A, B, or C?".

### How It Works

If your dataset has $N$ different classes, the OvR approach splits the problem into **$N$ separate binary classification problems**.

Let's use the **Iris dataset** from your first question as an example. It has 3 classes ($N=3$): Setosa, Versicolor, and Virginica.

**Step 1: Training**
Under the hood, the OvR approach trains 3 distinct binary classifiers:
1. **Classifier 1**: Is this flower **Setosa** or **Not Setosa** (Rest)?
2. **Classifier 2**: Is this flower **Versicolor** or **Not Versicolor** (Rest)?
3. **Classifier 3**: Is this flower **Virginica** or **Not Virginica** (Rest)?

During training for Classifier 1, all Setosa examples are labeled as `Class 1` (Positive), and all Versicolor AND Virginica examples are lumped together and labeled as `Class 0` (Negative). It repeats this logic for the other classifiers.

**Step 2: Prediction**
When you want to predict the species of a brand new flower, you pass that flower's data through **all 3** trained classifiers. 

Each classifier will output a confidence score or probability:
- Classifier 1 (Setosa?): 85% confident
- Classifier 2 (Versicolor?): 10% confident
- Classifier 3 (Virginica?): 5% confident

The OvR model simply looks at all the scores and picks the class from the classifier that is **the most confident**. In this case, the model would predict **Setosa**.

In [7]:
class OVRClassifier:
    def __init__(self, model_class, **kwargs):
        """
        kwargs: keyword arguments for the model:
            for LogisticRegression: lr, n_iter
            for SVM: C, lr, n_iter 
        """
        self.models = []
        self.classes = []
        self.model = model_class
        self.kwargs = kwargs           

    def fit(self, X, Y):
        self.models = []
        self.classes = np.unique(Y) 
        for cls in self.classes:
            ybinary = (Y == cls).astype(int)
            model = self.model(**self.kwargs)
            model.fit(X, ybinary)
            self.models.append(model)

    def predict(self, X):
        scores = np.array([model.prop(X) for model in self.models])
        return self.classes[np.argmax(scores, axis=0)]

    def score(self, X, Y):
        return np.mean(self.predict(X) == Y)

def accuracy(y_true, y_pred):
        return np.mean(y_true == y_pred)    

def confusion_matrix(y_true, y_pred):
        return np.array([[np.sum((y_true == i) & (y_pred == j)) 
                            for j in np.unique(y_pred)]
                             for i in np.unique(y_true)])

def _confusion_matrix(y_true, y_pred):
        classes = np.unique(np.concatenate([y_true, y_pred]))
        matrix = np.zeros((len(classes), len(classes)), dtype=int)
        for true, pred in zip(y_true, y_pred):  
            matrix[true][pred] += 1
        return matrix
        

In [ ]:
from sklearn.model_selection import train_test_split

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

ovr_lr = OVRClassifier(LogisticRegression, lr=0.1, n_iter=1000)
ovr_lr.fit(X_train, y_train)
y_pred_lr = ovr_lr.predict(X_test)

print("=== Logistic Regression ===")
print(f"Accuracy: {accuracy(y_test, y_pred_lr):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))


from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # use fit params from train only!

ovr_svm = OVRClassifier(SVM, C=1.0, lr=0.001, n_iter=1000)
ovr_svm.fit(X_train_scaled, y_train)
y_pred_svm = ovr_svm.predict(X_test_scaled)

print(f"SVM Accuracy (normalized): {accuracy(y_test, y_pred_svm):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))



=== Logistic Regression ===
Accuracy: 1.0000
Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
SVM Accuracy (normalized): 0.9667
Confusion Matrix:
[[10  0  0]
 [ 0  8  1]
 [ 0  0 11]]


---
---

# Non-statistical models 
are machine learning algorithms that make predictions without assuming any underlying probability distribution about the data. They don't model uncertainty — they just learn rules or boundaries directly from patterns.

### The key difference

| | Statistical Models | Non-Statistical Models |
|---|---|---|
| **Core idea** | Data comes from a probability distribution; estimate its parameters | Learn rules/patterns directly from data structure |
| **Output** | Often a probability (e.g. 72% chance of class A) | Usually a direct decision (class A or B) |
| **Assumption** | Makes assumptions about data distribution | Few or no distributional assumptions |
| **Examples** | Logistic Regression, Naive Bayes, Linear Regression | KNN, Decision Trees, Random Forests |

---

### Common non-statistical models

- **K-Nearest Neighbors (KNN)** — "What are the k closest training points? Vote on the label."
- **Decision Tree** — "Split data by the feature that separates classes best, recursively."
- **Random Forest** — Many decision trees, each trained on random subsets, vote together


---

## Decision Tree

**Description:** A Decision Tree recursively partitions the feature space by selecting the best feature and threshold to split on at each node. It continues splitting until a stopping condition is met (max depth, minimum samples, or pure node). Supports both classification and regression.

**How it works:**
1. At each node, evaluate every possible feature + threshold combination
2. Pick the split that maximizes the **gain** (reduction in impurity or variance)
3. Recursively repeat for the left and right children
4. Stop when: node is pure, max depth reached, or too few samples remain

**Classification — Gini Impurity:**

$$\text{Gini}(S) = 1 - \sum_{k=1}^{K} p_k^2$$

where $p_k$ is the proportion of class $k$ in set $S$.

**Classification — Information Gain:**

$$\text{Gain} = \text{Gini}(S) - \frac{|S_L|}{|S|}\text{Gini}(S_L) - \frac{|S_R|}{|S|}\text{Gini}(S_R)$$

**Regression — Variance Reduction:**

$$\text{Gain} = \text{Var}(S) - \frac{|S_L|}{|S|}\text{Var}(S_L) - \frac{|S_R|}{|S|}\text{Var}(S_R)$$

**Leaf Value:** Mode (most common class) for classification, Mean for regression.

**Prediction:** Traverse the tree from root to leaf, going left if $x_{\text{feature}} \leq \text{threshold}$, right otherwise.


In [ ]:
import numpy as np
from collections import Counter

class DecisionTree:
    class _DecisionTreeNode:
        def __init__(self):
            self.feature = None
            self.threshold = None
            self.left = None
            self.right = None
            self.label = None

        def predict(self, x):
            if self.label is not None:
                return self.label
            elif x[self.feature] <= self.threshold:
                return self.left.predict(x)
            else:
                return self.right.predict(x)   
                 
        def left_mask(self, X):
            return X[:, self.feature] <= self.threshold
            
        def right_mask(self, X):  
            return X[:, self.feature] > self.threshold          

    def __init__(self, min_samples_split=2, max_depth=100, task="classification"):
        self.head = self._DecisionTreeNode()
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        
        if task.lower() not in ["classification", "regression"]:
            raise ValueError("Task must be either 'classification' or 'regression'")
        self.task = task.lower()

    def fit(self, X, y, curr_node=None, depth=0): 
        if curr_node is None:
            curr_node = self.head 
            
        # Base case: pure node (only matters for classification really)
        if len(set(y)) == 1:
            curr_node.label = self._leaf_value(y)
            return
            
        # Base case: early stopping conditions
        if depth >= self.max_depth or len(y) < self.min_samples_split:
            curr_node.label = self._leaf_value(y)
            return    

        best_gain = -1
        best_threshold, best_feature = None, None
        
        for i in range(X.shape[1]):
            thresholds = np.unique(X[:, i])
            for threshold in thresholds:
                left_mask = X[:, i] <= threshold
                right_mask = X[:, i] > threshold

                # If the split creates an empty side, skip it
                if len(y[left_mask]) < 1 or len(y[right_mask]) < 1:
                    continue 

                # Use the abstracted routing metric to calculate Gain
                gain = self._calculate_gain(y, y[left_mask], y[right_mask])
                
                if gain > best_gain:
                    best_gain = gain
                    best_threshold = threshold
                    best_feature = i   
                    
        if best_gain > 0:                     
            curr_node.feature = best_feature
            curr_node.threshold = best_threshold
            curr_node.left = self._DecisionTreeNode()
            curr_node.right = self._DecisionTreeNode()
            
            # Recurse
            self.fit(X[curr_node.left_mask(X)], y[curr_node.left_mask(X)], curr_node.left, depth+1)
            self.fit(X[curr_node.right_mask(X)], y[curr_node.right_mask(X)], curr_node.right, depth+1)  
        else:
            # If no feature improves the gain, make it a leaf node
            curr_node.label = self._leaf_value(y)

    def predict(self, X):
        return np.array([self.head.predict(x) for x in X])
        
    
    def _leaf_value(self, y):
        """Returns Mean for regression, Mode for classification."""
        if self.task == "regression":
            return np.mean(y)
        else:
            return Counter(y).most_common(1)[0][0]

    def _calculate_gain(self, y, y_left, y_right):
        """Routes to either Variance Reduction or Information Gain"""
        p = len(y_left) / len(y)
        
        if self.task == "regression":
            # Variance Reduction
            def variance(arr): return np.var(arr) if len(arr) > 0 else 0
            return variance(y) - p * variance(y_left) - (1 - p) * variance(y_right)
            
        else:
            # Gini Information Gain
            def gini(arr):
                if len(arr) == 0: return 0
                _, counts = np.unique(arr, return_counts=True)
                return 1 - np.sum((counts / len(arr))**2)
            return gini(y) - p * gini(y_left) - (1 - p) * gini(y_right)


---

## Random Forest

**Description:** A Random Forest is an **ensemble** of Decision Trees, each trained on a different random subset of the data and features. By combining many diverse, slightly different trees, Random Forest reduces overfitting and improves generalization compared to a single Decision Tree.

**How it works:**
1. For each of $T$ trees (**Bagging**):
   - **Bootstrap** the rows: sample $m$ indices from $[0, m)$ with replacement → each tree sees ~63.2% unique rows
   - **Feature subsampling**: randomly select $\sqrt{n}$ features (without replacement) from $n$ total features
   - Train a `DecisionTree` on only the selected rows and features
   - Store which features were used (needed for prediction)
2. To predict, pass input through all $T$ trees using each tree's specific feature subset

**Bootstrap Sampling:**

$$\text{idxs} = \text{random\_choice}(m, m, \text{replace=True})$$

**Feature Subset Size:**

$$n_{\text{features}} = \lfloor\sqrt{n}\rfloor$$

**Aggregation:**
- **Classification:** Majority vote across all trees → $\hat{y} = \text{mode}(\hat{y}_1, \hat{y}_2, \ldots, \hat{y}_T)$
- **Regression:** Average across all trees → $\hat{y} = \frac{1}{T}\sum_{t=1}^{T}\hat{y}_t$


In [ ]:
class RandomForest:
    def __init__(self, n_estimators, max_depth, task="classification", min_samples_split=2):
        self.n_estimators = n_estimators
        self.trees = []
        self.features_per_tree = [] # We must remember the columns!
        self.max_depth = max_depth
        self.task = task
        self.min_samples_split = min_samples_split

    def fit(self, X, y):
        # Pick how many features to use (e.g., square root of total)
        n_features = int(np.sqrt(X.shape[1])) 
        
        for _ in range(self.n_estimators):
            # 1. Randomize Rows (Bootstrapping with replacement)
            idxs = np.random.choice(len(X), len(X), replace=True)
            
            # 2. Randomize Features (Subset WITHOUT replacement)
            features = np.random.choice(X.shape[1], n_features, replace=False)
            
            tree = DecisionTree(max_depth=self.max_depth, task=self.task, min_samples_split=self.min_samples_split)
            tree.fit(X[idxs][:, features], y[idxs])
            
            self.trees.append(tree)
            self.features_per_tree.append(features) # Save the feature list

    def predict(self, X):
        predictions = []
        for tree, features in zip(self.trees, self.features_per_tree):
            # We MUST pass the exact same subset of columns to the tree
            predictions.append(tree.predict(X[:, features]))
            
        if self.task.lower() == "regression":
            return np.mean(predictions, axis=0) 
        else:
            sample_predictions = np.array(predictions).T
            # For classification, we use voting
            final_votes = [Counter(row).most_common(1)[0][0] for row in sample_predictions]
            return np.array(final_votes)

---------------

In [ ]:
    
class XGBoost:
    def __init__(self, n_estimators, lr=0.1, max_depth=3, task="classification", min_samples_split=2):
        self.n_estimators = n_estimators
        self.lr = lr
        self.max_depth = max_depth
        self.task = task
        self.min_samples_split = min_samples_split

    def _softmax(self, z):
        # Shift z for numerical stability (doesn't change math, prevents overflow)
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        
        if self.task.lower() == "classification":
            y_onehot = np.eye(len(self.classes_))[y]
            self.trees = [[] for _ in self.classes_]
            
            # 1. Initialize Raw Scores (Logits) to 0
            logits = np.zeros((X.shape[0], len(self.classes_)))

            for _ in range(self.n_estimators):
                # 2. Convert logits to probabilities via Softmax
                probs = self._softmax(logits)
                
                for k in range(len(self.classes_)):
                    # 3. Calculate Gradient (Residual) = True - Prob
                    residual = y_onehot[:, k] - probs[:, k]
                    
                    # 4. Train REGRESSION tree on residuals (pseudo-gradients)
                    # Notice we hardcode "regression" here regardless of task!
                    tree = DecisionTree(max_depth=self.max_depth, task="regression", min_samples_split=self.min_samples_split)
                    tree.fit(X, residual)
                    
                    # 5. Update Logits for the next round
                    update = tree.predict(X)
                    logits[:, k] += self.lr * update
                    
                    self.trees[k].append(tree)
        elif self.task.lower() == "regression":
        
            self.trees = []
            self.base_pred = np.mean(y)
            preds = np.full(X.shape[0], self.base_pred)
            
            for _ in range(self.n_estimators):
                residual = y - preds
                tree = DecisionTree(max_depth=self.max_depth, task="regression", min_samples_split=self.min_samples_split)
                tree.fit(X, residual)
                preds += self.lr * tree.predict(X)
                self.trees.append(tree)

    def predict(self, X):
        if self.task.lower() == "classification":
            # Start with 0 logits
            logits = np.zeros((X.shape[0], len(self.classes_)))
            
            # Add up all tree predictions to get the final Raw Scores
            for k, trees_for_class in enumerate(self.trees):
                for tree in trees_for_class:
                    logits[:, k] += self.lr * tree.predict(X)
                    
            # Return original class label corresponding to the highest logit score
            # (Because Softmax is monotonic, the highest raw score is always the highest probability)
            return self.classes_[np.argmax(logits, axis=1)]
        elif self.task.lower() == "regression":
            preds = np.full(X.shape[0], self.base_pred)
            for tree in self.trees:
                preds += self.lr * tree.predict(X)
            return preds    


## XGBoost (Gradient Boosted Trees)

**Description:** XGBoost is a **sequential ensemble** method that builds trees one at a time, where each new tree corrects the mistakes of all previous trees. Unlike Random Forest (which builds trees independently in parallel), XGBoost fits trees to the **pseudo-residuals** (gradients of the loss function).

**How it works (Classification):**
1. Initialize raw scores (**logits**) to zero for each class
2. For each boosting round:
   - Convert logits to probabilities via **Softmax**
   - For each class $k$: compute residual = $y_{\text{true}} - p_k$, train a **Regression Tree** on those residuals
   - Update the logits: $\text{logits}_k \mathrel{+}= \alpha \cdot \text{tree\_prediction}$
3. Final prediction: $\arg\max$ of accumulated logits

**Softmax Function (with numerical stability):**

$$p_k = \frac{e^{z_k - \max(\mathbf{z})}}{\sum_{j} e^{z_j - \max(\mathbf{z})}}$$

**Pseudo-Residual (Negative Gradient of Cross-Entropy):**

$$r_{ik} = y_{ik}^{\text{onehot}} - p_{ik}$$

**Logit Update:**

$$z_k^{(t+1)} = z_k^{(t)} + \alpha \cdot h_k^{(t)}(X)$$

where $\alpha$ is the learning rate and $h_k^{(t)}$ is the regression tree fitted at round $t$ for class $k$.

**How it works (Regression):**
1. Initialize predictions to the mean of $y$
2. Each round: compute residual = $y - \hat{y}$, train a Regression Tree on the residuals
3. Update: $\hat{y} \mathrel{+}= \alpha \cdot \text{tree\_prediction}$

> **Key insight:** Even in classification, the internal trees are always **Regression Trees** because they predict continuous gradient values, not class labels.


In [ ]:
from collections import Counter
import numpy as np
import plotly.express as px

df = px.data.iris()
X = df[['sepal_length','sepal_width','petal_length','petal_width']].values
y = df['species'].map({'setosa':0,'versicolor':1,'virginica':2}).values
df 
# --- Train/Test Split ---
np.random.seed(42)
indices = np.random.permutation(len(X))
split = int(0.8 * len(X))
X_train, X_test = X[indices[:split]], X[indices[split:]]
y_train, y_test = y[indices[:split]], y[indices[split:]]

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred) * 100

# --- 1. Decision Tree ---
dt = DecisionTree(max_depth=5, task="classification")
dt.fit(X_train, y_train)
dt_preds = dt.predict(X_test)
print(f"Decision Tree  Accuracy: {accuracy(y_test, dt_preds):.1f}%")

# --- 2. Random Forest ---
rf = RandomForest(n_estimators=20, max_depth=5, task="classification")
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
print(f"Random Forest  Accuracy: {accuracy(y_test, rf_preds):.1f}%")

# --- 3. XGBoost ---
xgb = XGBoost(n_estimators=20, lr=0.3, max_depth=3, task="classification")
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
print(f"XGBoost        Accuracy: {accuracy(y_test, xgb_preds):.1f}%")

# --- Summary Table ---
print("\n" + "="*40)
print(f"{'Model':<18} {'Accuracy':>8}")
print("-"*40)
print(f"{'Decision Tree':<18} {accuracy(y_test, dt_preds):>7.1f}%")
print(f"{'Random Forest':<18} {accuracy(y_test, rf_preds):>7.1f}%")
print(f"{'XGBoost':<18} {accuracy(y_test, xgb_preds):>7.1f}%")
print("="*40)


# Are the predictions actually identical?
print("DT vs RF same predictions?", np.array_equal(dt_preds, rf_preds))
print("DT vs XGB same predictions?", np.array_equal(dt_preds, xgb_preds))
print("RF vs XGB same predictions?", np.array_equal(rf_preds, xgb_preds))

# Which samples did each model get wrong?
print("\nDT  wrong at indices:", np.where(dt_preds != y_test)[0])
print("RF  wrong at indices:", np.where(rf_preds != y_test)[0])
print("XGB wrong at indices:", np.where(xgb_preds != y_test)[0])



Decision Tree  Accuracy: 93.3%
Random Forest  Accuracy: 93.3%
XGBoost        Accuracy: 93.3%

Model              Accuracy
----------------------------------------
Decision Tree         93.3%
Random Forest         93.3%
XGBoost               93.3%
DT vs RF same predictions? False
DT vs XGB same predictions? True
RF vs XGB same predictions? False

DT  wrong at indices: [11 26]
RF  wrong at indices: [23 26]
XGB wrong at indices: [11 26]
